<h1>External Data Construction</h1>

This notebook is used to make the external datasets for the project in a way where we can compute the moments and use them for our model.

### imports

In [245]:
import pandas as pd
import numpy as np 
import nilearn
import glob
import os 
import matplotlib.pyplot as plt 
import seaborn as sns 
import warnings 


### helper functions

In [246]:
def power_mean_moments(V, K=10):
    """
    Robustly computes rooted raw moments up to order K.
    Handles negative bases for odd roots correctly.
    """
    n_paths, T_plus = V.shape
    M = np.empty((T_plus, K), dtype=np.float64)

    # Copy V to avoid modifying the input in place
    pow_k = V.copy()
    
    for k in range(1, K + 1):
        # 1. Compute the raw moment: E[x^k]
        mean_pow = pow_k.mean(axis=0)
        
        # 2. Compute the Rooted Moment: (E[x^k])^(1/k)
        if k == 1:
            M[:, k - 1] = mean_pow
        else:
            # FIX: Use abs() before root, then multiply by sign()
            # This handles odd roots of negative numbers (e.g., -8^(1/3) = -2)
            # Even roots of negative numbers would still be NaN, but E[x^even] is always >= 0, so it's safe.
            M[:, k - 1] = np.sign(mean_pow) * (np.abs(mean_pow) ** (1.0 / k))
        
        # 3. Prepare next power: V^(k+1) = V^k * V
        if k < K: # Optimization: Don't compute V^(K+1) on the last step
            pow_k *= V

    return M

## Dataset 1: S&P 500 Data

The first data that we will use is s&p 500 data. we will track the market fixed 500 companies from the s&p 500 index from 2020-2024. We will use the adjusted close price as our price for every ticker that was a part of the s&p 500 index at any point. 

We will then take all this data and perform the following preprocessing steps: 

1. Compute log returns for the adjusted close price 
1. Iterate through the data day by day and for each specific date t, we select active stocks and winsorize at fixed percentiles 
1. compute moments of the ~500 returns 

### process sp historical data

In [34]:
splist = pd.read_csv('../data/sp_historical_constituents.csv')
returns = pd.read_csv('../data/tickers_history.csv')

/var/folders/d0/4qvvbdzj6lg3k7zh204kqmk80000gn/T/ipykernel_76028/2750027044.py:2: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  returns = pd.read_csv('../data/tickers_history.csv')


In [40]:
def preprocess_data(constituents, returns): 
    
    #reformat some dates
    df_copy = pd.merge(constituents, returns, left_on='permno', right_on='PERMNO').copy() 
    df_copy['date'] = pd.to_datetime(df_copy['date'])
    df_copy['mbrstartdt'] = pd.to_datetime(df_copy['mbrstartdt'])
    df_copy['mbrenddt'] = pd.to_datetime(df_copy['mbrenddt'])
    df_copy['mbrenddt'] = df_copy['mbrenddt'].fillna(pd.Timestamp('2099-12-31'))

    #our filter
    mask = (df_copy['date'] >= df_copy['mbrstartdt'] ) & (df_copy['date'] <= df_copy['mbrenddt'])

    #merge returns and delisting returns
    df_copy['RET'] = pd.to_numeric(df_copy['RET'], errors='coerce').fillna(0)
    df_copy['DLRET'] = pd.to_numeric(df_copy['DLRET'], errors='coerce').fillna(0)

    # Merge the returns
    df_copy['RET'] = (1 + df_copy['RET']) * (1 + df_copy['DLRET']) - 1

    #rename columns
    df_copy = df_copy.drop(columns=['indno', 'mbrflg', 'indfam', 'PERMNO'])
    df_copy = df_copy.rename(columns={'DLRET':'dlret', 'RET':'ret'})

    return df_copy

In [71]:
#save new data 
new_data = preprocess_data(splist, returns)
new_data.to_csv('../data/sp500_historical_returns.csv', index=False)

### preprocess data

In [263]:
data = pd.read_csv('../data/sp500_historical_returns.csv')

In [264]:
data.head()

,permno,mbrstartdt,mbrenddt,date,TICKER,COMNAM,dlret,ret,sprtrn
0,10078,1992-08-20,2010-01-28,2000-01-03,SUNW,SUN MICROSYSTEMS INC,0.0,-0.012107,-0.009549
1,10078,1992-08-20,2010-01-28,2000-01-04,SUNW,SUN MICROSYSTEMS INC,0.0,-0.062092,-0.038345
2,10078,1992-08-20,2010-01-28,2000-01-05,SUNW,SUN MICROSYSTEMS INC,0.0,0.001742,0.001922
3,10078,1992-08-20,2010-01-28,2000-01-06,SUNW,SUN MICROSYSTEMS INC,0.0,-0.053913,0.000956
4,10078,1992-08-20,2010-01-28,2000-01-07,SUNW,SUN MICROSYSTEMS INC,0.0,0.056985,0.027090


In [265]:
data['gross_ret'] = (1 + data['ret']) * (1 + data['dlret']) -1 
data['log_ret'] = np.log1p(data['gross_ret'])

In [266]:
data = data.drop_duplicates(subset=['permno', 'date']) 

In [267]:
log_returns = data.pivot(index='date', columns='permno', values='log_ret')
log_returns = log_returns.sort_index()

In [268]:
log_returns.head()

permno,10078,10104,10107,10108,10137,10138,10145,10147,10225,10299,...,92988,93002,93089,93096,93132,93159,93246,93422,93429,93436
date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,-0.012181,0.052686,-0.001607,0.041243,-0.037830,-0.050314,-0.017487,0.042559,-0.024882,0.006095,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-04,-0.064103,-0.092510,-0.034364,0.010050,0.011976,-0.030716,-0.017798,-0.058708,-0.017596,0.027399,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,0.001740,-0.054261,0.010489,0.027129,0.044245,-0.001837,-0.013560,-0.042761,0.005900,0.046213,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-06,-0.055421,-0.060625,-0.034072,0.040530,-0.009154,0.028988,0.019155,-0.033316,0.009756,0.003221,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-07,0.055421,0.074015,0.012983,0.020810,0.002296,-0.007169,0.052185,0.074915,0.001940,0.053220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [269]:
dates = log_returns.index
moment_results = []

for t_idx, date in enumerate(dates):
    # Extract the cross-section for this day
    row = log_returns.iloc[t_idx].values
    
    # Remove NaNs (stocks not in index on this specific day)
    valid_returns = row[~np.isnan(row)]
    
    # Winsorization (Clip extreme data errors)
    if len(valid_returns) > 100: # Ensure we have a valid sample
        lower = np.percentile(valid_returns, 0.01 * 100)
        upper = np.percentile(valid_returns, 0.99 * 100)
        valid_returns = np.clip(valid_returns, lower, upper)
        
        # Reshape for our function (1, N_active)
        # The function expects shape (Time, Stocks), so we pass (1, N)
        V_t = valid_returns.reshape(1, -1)
        
        # Compute Moments
        m_t = power_mean_moments(V_t, K=10) # Returns shape (1, 10)
        moment_results.append(m_t[0])
    else:
        moment_results.append(np.full(10, np.nan))

# Compile Final DataFrame
moment_cols = [f'm{k}' for k in range(1, 11)]
final_df = pd.DataFrame(moment_results, index=dates, columns=moment_cols)

In [270]:
final_df.to_csv('../data/sp500_moments.csv', index=False)

## Dataset 2: FRMI Data

The second real-world dataset that we will use is the FRMI dataset. 

download the data

### import the data

In [ ]:
import nilearn.datasets as datasets
abide = datasets.fetch_abide_pcp(
    data_dir='./nilearn_data',
    pipeline='cpac',
    band_pass_filtering=True,
    global_signal_regression=False,
    quality_checked=True,
    derivatives=['rois_cc200'], 
    verbose=1
)
print(f"Downloaded {len(abide.rois_cc200)} subjects.")

[fetch_abide_pcp] Added README.md to ./nilearn_data
[fetch_abide_pcp] Dataset created in nilearn_data/ABIDE_pcp
[fetch_abide_pcp] Downloading data from https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Phenotypic_V1_0b_preprocessed1.csv ...
Downloaded 122880 of 449443 bytes (27.3%%,    2.8s remaining)
[fetch_abide_pcp]  ...done. (2 seconds, 0 min)

[fetch_abide_pcp] Downloading data from https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Outputs/cpac/filt_noglobal/rois_cc200/Pitt_0050003_rois_cc200.1D ...
[fetch_abide_pcp]  ...done. (1 seconds, 0 min)

[fetch_abide_pcp] Downloading data from https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Outputs/cpac/filt_noglobal/rois_cc200/Pitt_0050004_rois_cc200.1D ...
[fetch_abide_pcp]  ...done. (1 seconds, 0 min)

[fetch_abide_pcp] Downloading data from https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Outputs/cpac/filt_noglobal/rois_cc200/Pitt_0050005_rois_cc200.1D ...
[fetch_abide_pcp

### filter and preprocess abide data

In [154]:
metadata = pd.read_csv('../data/nilearn_data/ABIDE_pcp/subject_metadata.csv')

In [200]:
healthy = metadata[metadata['DX_GROUP'] == 2]
ad = metadata[metadata['DX_GROUP'] == 1]

In [219]:
def process_timeseries_region(region_idx, path, df): 
    '''
        takes the region index and dataframe of metadata as input 
        returns time series for a specified region of the brain
    '''
    roi_data_list = [] 
    for file_path in os.listdir(path): 
        file_id = file_path.split('.')[0]
        
        #not in the current df file ids 
        if file_id not in df['FILE_ID'].values: 
            continue
        
        # process the time series 
        try: 
            timeseries_df = pd.read_csv(os.path.join(path, file_path))

            if region_idx < timeseries_df.shape[1]: 
                roi_series = timeseries_df.iloc[:, region_idx].values
                roi_data_list.append(roi_series)
            else: 
                print(f"Region index {region_idx} out of bounds for file {file_path}")
        except Exception as e: 
            print(f"Error processing file {file_path}: {str(e)}")
    
    #return pandas df version of the frame 
    res = pd.DataFrame(roi_data_list)
    res = res.iloc[:, :176]
    res = res.dropna()
    return res



In [222]:
healthy_series_100 = process_timeseries_region(100, '../data/nilearn_data/ABIDE_pcp/csv',healthy)
ad_series_100 = process_timeseries_region(100, '../data/nilearn_data/ABIDE_pcp/csv', ad)

In [238]:
healthy_series_100.to_csv('../data/abide_healthy_series_100.csv', index=False)
ad_series_100.to_csv('../data/abide_ad_series_100.csv', index=False)

### compute moments

In [259]:
from scipy.stats import zscore
healthy_standard = pd.read_csv('../data/abide_healthy_series_100.csv')
ad_standard = pd.read_csv('../data/abide_ad_series_100.csv')

In [260]:
healthy_moments = power_mean_moments(healthy_standard.values)
ad_moments = power_mean_moments(ad_standard.values)

In [261]:
healthy_moments_df = pd.DataFrame(
    healthy_moments, 
    columns=['m1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7', 'm8', 'm9', 'm10']
)
ad_moments_df = pd.DataFrame(
    ad_moments, 
    columns=['m1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7', 'm8', 'm9', 'm10']
)

In [262]:
healthy_moments_df.to_csv('../data/abide_healthy_moments_100.csv', index=False)
ad_moments_df.to_csv('../data/abide_ad_moments_100.csv', index=False)

## Dataset 3: 